# 💬 生成式AI應用開發｜第05週｜150分鐘修訂版
## 手動對話紀錄、Streaming 與可重設聊天工具
承接第四周訊息列表 few-shot，將人工範例替換成實際 API 回答；銜接第七周最近五輪 history。
本檔另存修訂，原教材保留。先備：函式、list/dict、for、切片、role/content、函式作為參數。

> **學生版**：練習A的訊息組裝／reset、練習B串流迴圈保留TODO，類別封裝與重要事實為選讀。

## 🎯 學習目標
1. 分清人工 few-shot 範例與實際對話 history。
2. 以手動 user/assistant 配對支援追問，每輪重送 instructions。
3. 以 reset／最近N輪控制下一輪的輸入，觀察 usage。
4. 比較 response ID 與手動 history 的狀態保存方式。
5. 接收 delta、累積完整文字、確認 completed，失敗不寫入正式紀錄。
6. 用固定測試與實際請求內容驗證聊天行為。

## 🗺️ 150分鐘有效授課流程（休息另計）
| 分鐘 | 活動 | 學生成果 |
|---|---|---|
| 0–15 | 第四周銜接與獨立請求 | 區分範例與歷史 |
| 15–40 | 手動history、兩輪追問 | 訊息組裝、完成回答配對 |
| 40–55 | response ID比較 | 說明保存方式與取捨 |
| 55–80 | 必做A：reset、最近N輪 | 離線請求與有限視窗 |
| 80–105 | Streaming事件、模擬片段 | 看懂delta與completed |
| 105–135 | 必做B：串流與失敗保護 | 成功保存與失敗測試 |
| 135–150 | 固定測試、同儕檢查 | 紀錄與修正理由 |
類別、長期重要事實與互動CLI為延伸；不是本週必做門檻。
若API排障超過5分鐘，先用虛構回應完成離線契約檢查。
新增離線小示範與細節對照供理解，不需學生修改；仍以A、B及固定測試為核心，詳細語法／類別為選讀。

選讀僅保留本週新需求：**request參數展開、history副本／就地reset、usage計算與假client。第四周的getattr／or、集合與函式參數不重教；遇到不熟悉處可回第四周「回應檢查」「A4範例檢查」「A5固定測試」查閱。


## 1️⃣ 環境與成本
金鑰放Colab Secrets的OPENAI_API_KEY；本機用環境變數，勿提交金鑰。
無金鑰可完成函式定義、訊息檢查及假事件測試。所有付費旗標預設False。
安裝cell會連線安裝SDK，但不呼叫模型；後面的假事件測試不連線。

In [ ]:
# Colab runtime重新開啟時安裝SDK；本機也可使用已安裝的課堂環境。
!pip install -q --upgrade openai

In [ ]:
import os
from openai import OpenAI

# Colab 優先讀取 Secrets；本機執行時則改用既有環境變數。

try:
    from google.colab import userdata
    secret_key = userdata.get("OPENAI_API_KEY")
    if secret_key:
        os.environ["OPENAI_API_KEY"] = secret_key
except ImportError:
    # 非 Colab 環境會走到這裡，不需要中斷 notebook。
    pass
except Exception as exc:
    print(f"無法讀取 Colab Secret：{exc}")


# 沒有金鑰仍能做離線組裝與假事件練習；付費入口才檢查 client。
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
client = OpenAI() if os.getenv("OPENAI_API_KEY") else None
print("本週模型：", MODEL)
print("已讀取金鑰（不顯示）" if client else "離線模式：尚未設定金鑰")


### 🧭 共用程式的責任（已提供，不是練習）
| 函式／變數 | 輸入與回傳 | 在流程中的位置 |
|---|---|---|
| CHAT_INSTRUCTIONS | 固定規則str | 每輪作為instructions送出 |
| require_text／require_client | 檢查輸入／client；不合法就拋錯 | API之前，不產生請求 |
| validated_output | response→有效回答str | API之後，保存history之前 |
| create_response | 問題與可選前輪ID→response物件 | 獨立請求／ID比較，每次一個請求 |
前三個檢查函式不呼叫API。`create_response`回傳的是response物件，顯示文字要讀其output_text；後面的`ask_chat_once`才直接回傳回答str。
下列兩格按順序執行定義；不要把API helper改成學生練習答案。

In [ ]:
import json

CHAT_INSTRUCTIONS = """
# Identity
你是生成式 AI 應用開發課程的助教。
# Instructions
- 使用繁體中文，先直接回答，再補充必要說明。
- 回答控制在 150 字內；不知道時清楚說明限制。
- 不要聲稱記得目前對話以外的資料。
"""



def require_text(text, name="user_input"):
    """空白輸入在本機擋下，不產生付費請求。"""
    if not isinstance(text, str) or not text.strip():
        raise ValueError(f"{name} 必須是非空白字串。")


def require_client(api_client):
    if api_client is None:
        raise RuntimeError("請設定 OPENAI_API_KEY，重新執行環境 cell。")


def validated_output(response):
    """HTTP 請求成功仍可能沒有可用答案；完成、拒答、空文字要分開檢查。"""
    if response.status != "completed":
        raise RuntimeError(f"回應未完成：{response.status}；不保存部分內容。")
    for item in response.output or []:
        for part in getattr(item, "content", []) or []:
            if getattr(part, "type", None) == "refusal":
                raise RuntimeError("模型拒答；本輪不寫入對話紀錄。")
    require_text(response.output_text, "output_text")
    return response.output_text

### `response = active_client.responses.create(**request)`（細節選讀）

**對應下一個正式code cell的create_response函式。** request是先準備好的參數字典，`**request`在呼叫時把字典展開為「參數名稱=值」。它不是把整個字典當成一個位置參數傳入，也不是在這裡做次方運算。

| request中的key | 展開後的具名參數 | 正式程式中的值 |
|---|---|---|
| "model" | model=MODEL | 指定模型 |
| "instructions" | instructions=CHAT_INSTRUCTIONS | 每輪規則 |
| "input" | input=user_input | 本次問題 |
| "store" | store=store | 呼叫端指定的保存設定 |
| "previous_response_id"（有前輪ID才加入） | previous_response_id=previous_response_id | 串接前輪回應 |

例如，沒有前輪ID時：

```python
# 以下兩種呼叫傳入的參數相同；只是閱讀對照，不需另外執行API。
response = active_client.responses.create(**request)

# 展開後相當於：
response = active_client.responses.create(
    model=MODEL,
    instructions=CHAT_INSTRUCTIONS,
    input=user_input,
    store=store,
)
```

**為什麼原程式先建立字典？** `request = dict(...)`準備共用參數；若previous_response_id有值，再加一個key。呼叫仍維持同一行，第一次請求不帶前輪ID、追問時才帶入，不必寫兩份幾乎相同的API呼叫。

注意字典key須為字串，且API必須接受該參數名稱；不是任意字典都能傳入。若同時寫`create(model=MODEL, **request)`且request已含model，會因參數重複而拋出TypeError。

**下一格離線示範與正式程式的對照：**

| 離線小示範 | 正式程式 |
|---|---|
| toy_request_parameters | request參數字典 |
| toy_show_request(**toy_request_parameters) | active_client.responses.create(**request) |
| 加入虛構previous_response_id | if previous_response_id成立時增加參數 |
| 回傳dict供assert觀察 | 正式API回傳response物件，還須validated_output檢查 |

小示範只顯示Python收到的參數，不呼叫API。先觀察第一筆前輪ID為None，再觀察加入key後變成虛構ID；回到create_response找出建立字典、條件加入、展開呼叫三個位置。

後面的假client還會用到`def create(**kwargs)`：函式定義中的`**kwargs`是**收集**具名參數成字典，呼叫端的`**request`則是**展開**字典；兩者方向相反。


In [ ]:
# 細節選讀：對照create_response的request與**request；只驗證Python參數傳遞，不連線。
def toy_show_request(model, instructions, input, store, previous_response_id=None):
    # 這裡接收的是各個具名參數，不是單一request字典。
    received = dict(model=model, instructions=instructions, input=input, store=store,
                    previous_response_id=previous_response_id)
    print("離線函式收到：", received)
    return received


# key需對應函式的參數名稱；model只是虛構字串，並未選用或呼叫真實模型。
toy_request_parameters = {
    "model": "offline-demo",
    "instructions": "只顯示收到的參數。",
    "input": "活動在哪裡？",
    "store": False,
}
# **在呼叫時展開字典：等同model="offline-demo", instructions=..., input=..., store=False。
toy_first_received = toy_show_request(**toy_request_parameters)
assert toy_first_received["input"] == "活動在哪裡？"
assert toy_first_received["previous_response_id"] is None

# 對照正式函式的條件加參數：加入key後，同一個**呼叫就會一併傳入它。
toy_request_parameters["previous_response_id"] = "fictional-previous-id"
toy_second_received = toy_show_request(**toy_request_parameters)
assert toy_second_received["previous_response_id"] == "fictional-previous-id"


In [ ]:
def create_response(user_input, previous_response_id=None, store=True, api_client=None):
    """供獨立請求與 response ID 比較示範；主線使用下方手動 history。"""
    require_text(user_input)
    active_client = client if api_client is None else api_client
    require_client(active_client)
    # 細節選讀對照：先建立具名參數字典，前輪ID存在時才加入額外key。
    request = dict(model=MODEL, instructions=CHAT_INSTRUCTIONS, input=user_input, store=store)
    if previous_response_id:
        request["previous_response_id"] = previous_response_id
    try:
        # **request把字典展開為具名參數；此處才發出一次API請求，不是將字典當單一input。
        response = active_client.responses.create(**request)
    except Exception as exc:
        raise RuntimeError(f"API 請求失敗：{exc}") from exc
    validated_output(response)
    return response

### 共用回應檢查：接回第四周

第四周request_text已示範完成狀態、拒答與非空文字檢查；第五周抽成validated_output，讓非串流與串流完成後共用，不重教getattr／or語法。
**對照ask_chat_once的兩行：** `answer = validated_output(response)` → `save_completed_turn(...)`。檢查拋錯時就不保存本輪配對；stream_response也須先驗證完成response，才能交給外層保存。


### 從第四周到第五周
| 項目 | 第四周 | 第五周 |
|---|---|---|
| assistant內容 | 人工指定範例答案 | API實際完成的回答 |
| 用途 | 示範任務模式 | 提供前文以支援追問 |
| 保存 | 固定範例 | 隨互動增加的本機history |
規則仍放instructions；看到Notebook有舊輸出，不代表模型收到它。
先預測第二輪要看到哪些訊息，再印出送出的input。

## 2️⃣ 獨立請求：為何忘記前文？
兩次請求若沒有附history或response ID，第二次看不到第一輪。
觀察是否拒絕猜測姓名；即使碰巧猜對，也不能證明具有記憶。

In [ ]:
# 獨立請求示範：先設定金鑰；2次API請求，確認後才開啟。
RUN_STATELESS_DEMO = False
if RUN_STATELESS_DEMO:
    first = create_response("我叫小安，專題是校園活動推薦。", store=False)
    second = create_response("我的姓名與專題是什麼？", store=False)
    print(first.output_text)
    print(second.output_text)

## 3️⃣ 手動history：先做最小的兩輪流程
第一輪只送user；收到有效完成回答後，保存user與assistant。
第二輪把已完成配對加上新user送入input。固定instructions每輪提供。
下列最小示範先不裁切，避免一次加入太多概念；練習A再實作有限視窗。
本週為單一文字回覆的簡化history；需要保留推理／工具項目的正式流程應保存必要的完整output項目。

### 先用離線配對看懂history（示範，不需修改）
下格用虛構的校園活動問答，不呼叫模型。它對應接下來RUN_MANUAL_DEMO中的三行：
| 小示範 | 正式示範 |
|---|---|
| toy_history | demo_history，只放已完成user/assistant配對 |
| toy_request = toy_history + [...] | request_input；`+`產生新列表，本次user暫時不寫入history |
| toy_history.extend([...]) | 完成回答驗證後才保存這一輪配對 |
先預測第二輪request有幾則，再看列印：應為3則；儲存第二輪後history才變成4則。
虛構toy_answer不是模型回答；正式示範的answer來自validated_output(response)。

In [ ]:
# 語法選讀：對照RUN_MANUAL_DEMO的request_input與demo_history；下列回答是虛構資料。
toy_history = [
    {"role": "user", "content": "活動在哪裡？"},
    {"role": "assistant", "content": "圖書館一樓。"},
]
toy_question = "需要報名嗎？"
# 用+建立本輪input，附上新user但不先改history；正式API失敗時不應留下半輪紀錄。
toy_request = toy_history + [{"role": "user", "content": toy_question}]
print("第二輪input：", toy_request, "｜原history則數：", len(toy_history))
toy_answer = "這是虛構示範：需要先報名。"
# 正式程式必須等API回答通過檢查，才做下列保存。
# extend加入兩個dict，等價於正式示範通過validated_output後保存user與assistant。
toy_history.extend([
    {"role": "user", "content": toy_question},
    {"role": "assistant", "content": toy_answer},
])
# 第二輪input為舊配對2則＋新user1則；保存成功後history才增加到4則。
assert len(toy_request) == 3 and len(toy_history) == 4
print("保存後history則數：", len(toy_history))

In [ ]:
# 手動history示範：先設定金鑰；2次API請求，確認後才開啟。
RUN_MANUAL_DEMO = False
if RUN_MANUAL_DEMO:
    require_client(client)
    demo_history = []
    for question in ["我叫小安，專題是校園活動推薦。", "請重述我的姓名與專題。"]:
        request_input = demo_history + [{"role": "user", "content": question}]
        print("本輪實際input：", request_input)
        response = client.responses.create(
            model=MODEL, instructions=CHAT_INSTRUCTIONS, input=request_input, store=False,
        )
        answer = validated_output(response)
        demo_history.extend([
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ])
        print(answer)

### ✍️ 預測與觀察
| 回合 | 預測input訊息數 | 實際input | 是否依前文作答 | 有無編造 |
|---|---|---|---|---|
| 第一輪 |  |  |  |  |
| 第二輪 |  |  |  |  |
判讀：第一輪1則，第二輪3則；第二輪assistant是第一輪真正收到的答案。
回答措辭可以不同，姓名／主題須符合前文。

## 4️⃣ response ID作比較（不另寫一套完整聊天工具）
| 方法 | 本機保存 | 每輪傳送 | 本週定位 |
|---|---|---|---|
| 手動history | 完成的訊息配對 | 最近N輪＋本次問題 | 核心實作、第七周銜接 |
| previous_response_id | 前輪response.id | 前輪ID＋本次問題 | 比較示範 |
| Conversations API | conversation ID | 依API管理持久對話 | 概念選讀 |
response ID串接仍每輪提供instructions；鏈中舊輸入仍可能計入input tokens。
本比較採store=True；本機reset不等於刪除平台已保存response。
store=False也不代表資料沒有傳送或處理；只使用虛構資料。

In [ ]:
# response ID比較：先確認store=True保存方式；2次API請求，確認後才開啟。
RUN_ID_COMPARISON = False
if RUN_ID_COMPARISON:
    id_first = create_response("我叫小安，專題是校園活動推薦。", store=True)
    id_second = create_response("請重述姓名與專題。", previous_response_id=id_first.id, store=True)
    print(id_first.id, "→", id_second.id)
    print(id_second.output_text)

## 5️⃣ 必做A：本機state、最近N輪與reset（55–80分）
**情境**：聊天逐漸變長，每輪只送最近N輪；使用者reset後，下一次發問不帶舊資料。
**已提供**：new_chat_state、history_size、save_completed_turn、ask_chat_once與測試器。
**學生負責**：build_chat_input與reset_chat；不用重寫API入口或自行呼叫save_completed_turn。
| 任務 | 具體工作 | 完成條件 |
|---|---|---|
| A1a | 閱讀已提供的文字／max_turns驗證；可加強history配對檢查 | max_turns須為正整數，history以完整配對為前提 |
| A1b | 選history最近2*max_turns則 | 不拆開user/assistant一輪 |
| A1c | 複製role/content到新訊息dict | 改input內容不能影響原history |
| A1d | 加本次user、return新列表 | 空history為1則；最近1輪為3則 |
| A2 | 清空原state，再以new_chat_state更新 | 原dict仍是同一物件，history／usage／成功數重設 |
**操作**：執行共用定義→完成A1/A2→重跑函式定義→RUN_INPUT_CHECK改True並重跑→通過後才開RUN_PRACTICE_A。
離線檢查0次API；付費A用兩輪追問＋reset後一輪，共3次。
`max_turns=5`最多帶10則已完成訊息＋本次user。裁切控制input，本機history仍保留全部；reset才清空本機狀態，不刪除平台紀錄或帳單。

### state欄位與資料流（共用程式，不需補TODO）
| state欄位 | 型別 | 何時改變？ |
|---|---|---|
| history | list[dict]，已完成訊息配對 | 有效回答後加入兩則；reset清空 |
| usage | dict，三個token累計 | 只累計已接受的成功回合 |
| completed_requests | int | 每個成功回合加1；reset歸零 |
下一格建立state、觀察字元數；再下一格save_completed_turn保存回答與統計。它不呼叫API，也不驗證回答，呼叫端必須先驗證。
`history_size`的字元數不是token。usage缺少時helper以0累計；不能據此推論該請求免費。
`history_size`中的`sum(len(m["content"]) for m in messages)`逐筆計算content長度後加總，可讀成一般for累計；只觀察字元規模，不需另外改寫生成器。


In [ ]:
def new_chat_state():
    """本機保存已完成 history、成功回合使用量與成功請求數。"""
    return {
        "history": [],
        "usage": {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0},
        "completed_requests": 0,
    }


def history_size(messages):
    """字元數方便觀察，不能當成精準 token 數或費用。"""
    # 細節選讀：生成器逐筆提供content字元數給sum，不建立中間列表；不是token統計。
    return {"訊息數": len(messages), "總字元數": sum(len(m["content"]) for m in messages)}

In [ ]:
def save_completed_turn(state, user_text, answer, response):
    """所有回應檢查通過後才一次寫入 user/assistant 配對。"""
    # 先準備 usage，避免格式讀取錯誤後留下只更新一半的 state。
    usage = getattr(response, "usage", None)
    # 細節選讀對照：字典生成式逐個key計算新累計；先完成計算，再一起更新history與usage。
    new_usage = {
        key: state["usage"][key] + (getattr(usage, key, 0) or 0)
        for key in state["usage"]
    }
    state["history"].extend([
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": answer},
    ])
    state["usage"] = new_usage
    state["completed_requests"] += 1

### usage如何累計？（細節選讀，對照save_completed_turn）

第四周已說明getattr／or，不重複解釋。本週新增的是new_usage字典生成式：`for key in state["usage"]`依序取得input_tokens、output_tokens、total_tokens的key，再用「原累計＋本次用量」建立新dict。
若不熟悉生成式，可讀成先建空dict，再用for逐個key設定加總結果；學生不需修改這個共用函式。
先算好new_usage，才保存user／assistant配對與更新統計，避免讀用量出錯後留下半套state。最後completed_requests加1只計成功回合；缺usage以0處理，不代表請求免費。


### A的函式規格與離線語法對照
| 函式 | 輸入 | 回傳／副作用 |
|---|---|---|
| build_chat_input(user_text, history, max_turns) | 問題str、完整配對list、正整數 | 新list[dict]；不修改history |
| reset_chat(state) | new_chat_state建立的dict | 就地修改，回傳None；不要用state = reset_chat(state) |
下一格用字串列表看切片：每輪兩項，所以最近一輪取最後2項。對應A1b；正式訊息是dict，A1c還須複製dict，只有列表切片不能隔離dict內容。
reset中的`state.clear()`與`state.update(...)`改原物件。若函式內只寫`state = new_chat_state()`，只是把區域變數指向另一個dict，外面的state不會清空。
下面只示範切片與物件關係；學生仍須自行完成正式訊息組裝及reset。
教師參考的enumerate提供索引與訊息，索引從0開始；index % 2用於判斷user／assistant交錯。這是配對驗證提示，不另設選讀示範，也可用一般for完成。


In [ ]:
# 語法選讀：對照A1最近N輪與A2就地reset；只示範資料操作，不提供完整練習答案。
toy_turns = ["問1", "答1", "問2", "答2", "問3", "答3"]
# 每輪問答兩項，-2從倒數第二項取到結尾；正式history改用2*max_turns計算。
print("最近一輪：", toy_turns[-2:], "｜原列表：", toy_turns)
# 切片產生新列表，但裡面的dict仍可能是同一物件。
toy_records = [{"content": "原值"}]
toy_shallow = toy_records[:]
# 故意改切片中的dict，觀察原列表也變更：A1c必須建立新的訊息dict才可隔離。
toy_shallow[0]["content"] = "共用dict已變更"
assert toy_records[0]["content"] == "共用dict已變更"
print("切片仍共用內部dict：", toy_records)
toy_settings = {"theme": "dark"}
# 兩個名稱指向同一dict；接下來clear／update修改該物件，所以兩個名稱都看得到變更。
toy_alias = toy_settings
toy_settings.clear()
toy_settings.update({"theme": "light"})
# 對照reset_chat：要保留原state身分，不能只在函式內把state指向另一個新dict。
assert toy_alias is toy_settings and toy_alias["theme"] == "light"
print("就地更新後，另一個引用也看得到：", toy_alias)

In [ ]:
def build_chat_input(user_text, history, max_turns=5):
    """A1：只選最近完整N輪，再附上本次user，不修改原history。"""
    require_text(user_text)
    if type(max_turns) is not int or max_turns <= 0:
        raise ValueError("max_turns必須是正整數。")
    # TODO A1a：已提供user_text/max_turns驗證；可加強history偶數長度、角色交錯與非空content檢查。
    # TODO A1b：選取history最近2*max_turns則，保持完整配對。
    # TODO A1c：為每則建立新的role/content dict，不與原history共用dict。
    # TODO A1d：在新列表附上本次user，return此列表；不先保存到state。
    # 提示：history中每輪都是user/assistant配對；不能直接append到原列表。
    raise NotImplementedError("請完成最近N輪組裝。")


def reset_chat(state):
    """A2：清空本機state，原dict物件仍可被介面使用。"""
    # TODO A2：clear state，再用new_chat_state()的內容更新。
    raise NotImplementedError("請完成reset。")

### A完成後：沿著非串流入口讀一次
**對應下一格ask_chat_once**：build_chat_input組裝→create一次API→validated_output取得回答→save_completed_turn更新state→return answer。
build_chat_input的回傳值存入messages，才放進API input；instructions每輪重送。
新user還不在state裡，若API或回應驗證拋錯，保存函式不會執行；原state因此保留。
下一格程式完整提供，學生只補A1/A2。先跑後面的RUN_INPUT_CHECK（離線），通過後才做付費三輪。

In [ ]:
def ask_chat_once(state, user_text, max_turns=5, instructions=CHAT_INSTRUCTIONS, api_client=None):
    """手動 history 的一次問答；完成後保存，失敗不改 state。"""
    require_text(user_text)
    require_text(instructions, "instructions")
    messages = build_chat_input(user_text, state["history"], max_turns)
    active_client = client if api_client is None else api_client
    require_client(active_client)
    try:
        response = active_client.responses.create(
            model=MODEL, instructions=instructions, input=messages, store=False,
        )
    except Exception as exc:
        raise RuntimeError(f"聊天失敗，狀態未更新：{exc}") from exc
    answer = validated_output(response)
    save_completed_turn(state, user_text, answer, response)
    return answer

### A的檢查與預期結果（測試器已提供）
RUN_INPUT_CHECK預設False；完成A後改True並重跑下格，不需要金鑰。
sample_history四則，max_turns=1應保留最後兩則加本次user，共3則；sample_history仍為4則。
scratch重設後應等於new_chat_state：history空列表、usage三項0、completed_requests為0。
assert失敗先核對A1/A2，不刪assert或改預期資料。輸入副本是否與原history隔離，會在後續補充檢查中確認。
通過後開RUN_PRACTICE_A：第一輪input1則／history2則；第二輪input3則／history4則；reset後發問input1則，新session成功數從1重新累計。

In [ ]:
RUN_INPUT_CHECK = False
if RUN_INPUT_CHECK:
    sample_history = [
        {"role": "user", "content": "我叫小安"}, {"role": "assistant", "content": "你好小安"},
        {"role": "user", "content": "我喜歡Python"}, {"role": "assistant", "content": "了解"},
    ]
    trimmed = build_chat_input("我叫什麼？", sample_history, max_turns=1)
    assert trimmed == sample_history[-2:] + [{"role": "user", "content": "我叫什麼？"}]
    assert len(sample_history) == 4
    print("真正送入的input：", trimmed)
    print("規模：", history_size(trimmed))
    scratch = new_chat_state()
    scratch["history"] = sample_history.copy()
    reset_chat(scratch)
    assert scratch == new_chat_state()

In [ ]:
# 練習A：先完成A與RUN_INPUT_CHECK；3次API請求，確認後才開啟。
RUN_PRACTICE_A = False
if RUN_PRACTICE_A:
    practice_state = new_chat_state()
    for question in ["我叫小安，專題是課程FAQ。", "我的專題是什麼？"]:
        print("本輪input：", build_chat_input(question, practice_state["history"], 5))
        print(ask_chat_once(practice_state, question))
    print("累積usage：", practice_state["usage"])
    reset_chat(practice_state)
    print("reset後input：", build_chat_input("我的專題是什麼？", practice_state["history"]))
    print(ask_chat_once(practice_state, "我的專題是什麼？"))

### ✍️ 狀態觀察表
| 操作 | 本機history則數 | 實際input則數 | 成功請求數 | usage | 判讀 |
|---|---|---|---|---|---|
| 第一輪 |  |  |  |  |  |
| 第二輪 |  |  |  |  |  |
| reset後 |  |  |  |  |  |
| 裁切最近1輪 |  |  |  |  |  |
「送出的資料不含姓名」是reset／裁切的證據；模型猜對或猜錯本身不是可靠證據。
usage只累計成功回合，不代表所有帳單費用；失敗／中斷可能仍有費用，須核對平台用量。

## 6️⃣ Streaming：先模擬，再寫事件迴圈
| 事件 | 做什麼 | 是否寫入正式history |
|---|---|---|
| response.created | 知道串流開始 | 否 |
| response.output_text.delta | 顯示片段、累積文字、增加計數 | 否 |
| response.completed | 驗證完成文字、取得usage | 通過檢查後才保存 |
| error／response.failed／response.incomplete | 提示失敗、停止 | 否 |
| 結束卻沒有completed | 視為未完成 | 否 |
delta數不是token數，也不是字數。收到片段不表示整輪成功。
畫面可能已有部分文字；失敗時須標示中斷，不能把它當完整答案保存。
對照下一格：`print(delta, end="", flush=True)`連續顯示片段；`"".join(collected)`依序接成字串。這兩行服務串流顯示／累積，不代表已完成或已保存。


In [ ]:
# 不付費的小示範：迴圈同時顯示與累積，不需要callback或nonlocal。
# 對照stream_response的event.delta：這裡直接用片段列表，不模擬完整API事件。
simulated_chunks = ["你好", "，", "我可以幫忙。"]
collected = []
delta_count = 0
for delta in simulated_chunks:
    # 對照print_delta：end避免每片段換行，flush要求立即送出顯示，並非立即保存history。
    print(delta, end="", flush=True)
    # 對照chunks.append(event.delta)：保留原順序，稍後join才取得完整字串。
    collected.append(delta)
    # 對照chunk_count：每個片段加1；片段數不是token數，也不是字元數。
    delta_count += 1
# 本格沒有completed事件，僅展示累積方式；正式程式還須完成與拒答檢查才能保存。
print("\n完整文字：", "".join(collected), "｜片段數：", delta_count)

### 接回第四周：把顯示函式交給串流器

第四周A5把分類函式交給測試器；本週相同概念是`on_delta=print_delta`或`on_delta=displayed.append`，收到delta才執行`callback(event.delta)`，不再另做函式物件小示範。
**對照stream_response：** callback只負責顯示／收集片段；chunks與chunk_count由事件迴圈保存及計數。正式history仍須等完成驗證後更新，不能在callback裡提前保存。


## 7️⃣ 必做B：串流事件與成功保存（105–135分）
**情境**：邊收到片段邊顯示，整輪完成且驗證通過才保存正式對話。
**已提供**：print_delta、參數驗證、chunks／chunk_count／completed_response初值、callback選擇、ask_chat_stream與假事件測試器。
**學生負責**：stream_response中的事件迴圈與完成後檢查；不改保存函式。
| TODO | 具體項目 | 完成條件 |
|---|---|---|
| B1 | 呼叫active_client.responses.create並用for讀事件 | stream=True、store=False；每次函式一個請求 |
| B2 | delta分支：累積、加1、呼叫callback | 片段依原順序保留，不在此更新state |
| B3 | completed保存event.response；error／failed／incomplete拋錯 | 非文字事件如created可忽略 |
| B4 | 迴圈結束必須有completed_response，交validated_output驗證 | 缺完成、拒答、空文字都不能成功 |
| B5 | join chunks與完成文字比較，回傳三個值 | (完整str, response物件, delta數int)，文字不一致拋錯 |
請用try/except把請求、事件迭代及callback的失敗轉為RuntimeError，讓呼叫端一致辨識未成功。
**操作**：完成B1–B5並重跑定義→執行假回應／假事件定義→開RUN_FAKE_STREAM_CHECK→通過後開RUN_OFFLINE_CHECKS→最後才開付費RUN_PRACTICE_B（2次請求）。
教師可用RUN_STREAM_DEMO示範1次；學生版也依賴A/B已完成，未完成時可看前面的離線片段示範。

In [ ]:
def print_delta(delta):
    """具名 callback：只顯示片段，不負責保存正式對話紀錄。"""
    # 細節選讀：end取消自動換行，flush刷新輸出；畫面片段尚不是正式history。
    print(delta, end="", flush=True)


def stream_response(messages, instructions=CHAT_INSTRUCTIONS, on_delta=None, api_client=None):
    """手動 history 串流入口；回傳完整文字、完成response、delta數。"""
    require_text(instructions, "instructions")
    if not isinstance(messages, list) or not messages:
        raise ValueError("messages 必須是非空列表。")
    active_client = client if api_client is None else api_client
    require_client(active_client)
    chunks = []
    chunk_count = 0
    completed_response = None
    # 函式參數的呼叫方式與第四周的測試器相同；不需要 nonlocal。
    # 沿用第四周函式參數：此處選擇函式物件，收到delta時才呼叫callback。
    callback = print_delta if on_delta is None else on_delta

    # TODO B1：啟動 stream=True、store=False，逐筆讀取 event。
    # TODO B2：delta 時累積文字、增加 chunk_count、呼叫 callback。
    # TODO B3：completed 保存 event.response；error/failed/incomplete 拋出 RuntimeError。
    # TODO B4：沒有 completed 時拒絕；用 validated_output 檢查完成回答。
    # TODO B5：join chunks，確認與完成文字相同，再回傳三個值。
    # 提示：非文字事件可忽略；任何失敗都不能回傳部分文字當作成功。
    raise NotImplementedError("請完成串流事件迴圈。")

### B的回傳值與保存位置（對照stream_response→ask_chat_stream）
| 層次 | 回傳 | 是否修改state？ |
|---|---|---|
| stream_response | full_text、完成response、chunk_count三個值 | 不接收state，不保存history |
| ask_chat_stream | full_text、chunk_count兩個值 | 驗證成功後呼叫save_completed_turn |
| 執行區text, count = ask_chat_stream(...) | 接收文字與片段數，顯示觀察 | 使用stream_state保存兩輪 |
下一格的`full_text, response, chunk_count = stream_response(...)`是三值解包；若B只回傳str，這一行就不符合規格。
發生例外時不會進入下一行save_completed_turn，history／usage／成功數不變；但callback已顯示或收集的片段可能存在，畫面需標示中斷。
程式核對完整文字不等於查證內容正確，姓名／專題仍需人工核對。


In [ ]:
def ask_chat_stream(state, user_text, max_turns=5, instructions=CHAT_INSTRUCTIONS,
                    on_delta=None, api_client=None):
    """重用非串流的訊息組裝及成功保存邏輯，只有取得回答的方式不同。"""
    messages = build_chat_input(user_text, state["history"], max_turns)
    full_text, response, chunk_count = stream_response(
        messages, instructions, on_delta=on_delta, api_client=api_client,
    )
    save_completed_turn(state, user_text, full_text, response)
    return full_text, chunk_count

In [ ]:
# 串流示範：先完成A/B；1次API請求，確認後才開啟。
RUN_STREAM_DEMO = False
if RUN_STREAM_DEMO:
    demo_state = new_chat_state()
    demo_text, demo_count = ask_chat_stream(demo_state, "用兩句話說明API。")
    print("\n完整文字：", demo_text, "delta數：", demo_count)
    print("正式history：", demo_state["history"])


In [ ]:
# 練習B：先完成A/B及離線檢查；2次API請求，確認後才開啟。
RUN_PRACTICE_B = False
if RUN_PRACTICE_B:
    stream_state = new_chat_state()
    for question in ["我叫小安，專題是課程FAQ。", "請重述我的姓名與專題。"]:
        print("本輪input：", build_chat_input(question, stream_state["history"]))
        text, count = ask_chat_stream(stream_state, question)
        print("\n片段數：", count, "｜完整字元數：", len(text))
    print("usage：", stream_state["usage"])

## 8️⃣ 固定測試：先看契約，再看模型品質
測試程式完整提供，不需補TODO；不改測試資料來讓結果通過。
**先備**：已完成A/B並重跑定義；依序執行fake_response、make_fake_client、make_fake_events的定義。
先跑RUN_FAKE_STREAM_CHECK觀察一段成功串流，再開RUN_OFFLINE_CHECKS確認五項契約：前文帶入、reset、最近N輪、空白不呼叫、串流完成才保存。
假回應驗證程式流程，不能證明模型會正確回答；另開RUN_LIVE_TESTS才會發出三次真實請求，以input和回答人工覆核。
假測試的requests只記錄「程式會送什麼」，不是網路請求；不需API key，也不付費。

### 假client如何接到正式程式？（細節選讀）

**對應下一格三個helper與run_offline_checks。** 第四周已用SimpleNamespace示範屬性讀取；本週用它建立正式函式需要的欄位，不重教點號或None。

| helper | 測試用途 |
|---|---|
| fake_response | 提供status／output／output_text／usage，交給正式驗證與保存函式 |
| make_fake_client(payload, requests) | 提供responses.create；呼叫時記錄參數，再回傳payload，不連線 |
| make_fake_events(kind) | 提供delta與完成／失敗事件；missing故意缺完成事件 |

傳入`api_client=fake`時，正式入口使用指定假client；未傳才用環境中的client。
`make_fake_client`內的create保留對外層payload與requests的引用，外層return後仍可使用；這裡只需理解「記錄收到的參數、回傳指定資料」，學生不需重寫閉包。
`def create(**kwargs)`將具名參數收集成dict，與第一部分`**request`展開方向相反。requests.append(kwargs)用於核對input，不是真實網路請求；假usage也不代表真實費用。


In [ ]:
# 細節選讀：對照validated_output、stream_response與run_offline_checks；以下都是離線虛構物件。
from types import SimpleNamespace


def fake_response(text="你好！", status="completed", refusal=False):
    """虛構回應只檢查程式流程，無法驗證模型品質或真實SDK行為。"""
    # 建立output/content/type結構，讓validated_output能沿正式欄位路徑檢查拒答。
    content = [SimpleNamespace(type="refusal")] if refusal else []
    return SimpleNamespace(
        id="offline-response", status=status, output_text=text,
        output=[SimpleNamespace(content=content)],
        usage=SimpleNamespace(input_tokens=10, output_tokens=3, total_tokens=13),
    )


def make_fake_client(payload, requests):
    # 用具名create函式替代SDK請求入口；不需要建立類別或使用lambda／nonlocal。
    def create(**kwargs):
        # **kwargs接收正式入口送來的model／instructions／input等具名參數；只記錄，不連線。
        requests.append(kwargs)
        return payload
    # create沒有括號，放入函式物件；正式程式呼叫fake.responses.create(...)才執行上方函式。
    return SimpleNamespace(responses=SimpleNamespace(create=create))


def make_fake_events(kind="completed"):
    events = [
        SimpleNamespace(type="response.created"),
        SimpleNamespace(type="response.output_text.delta", delta="你好"),
        SimpleNamespace(type="response.output_text.delta", delta="！"),
    ]
    # 對照串流B3/B4：成功事件攜帶完整response；missing不補完成事件，用來測未完成保護。
    if kind == "completed":
        events.append(SimpleNamespace(type="response.completed", response=fake_response()))
    elif kind != "missing":
        events.append(SimpleNamespace(type=kind))
    return events

### B先做一筆假串流（完成A/B後開啟，0次API）
下格直接使用正式ask_chat_stream，故也會呼叫build_chat_input；A未完成時先回A。
預期：文字「你好！」、delta數2、shown保存兩個片段、正式history新增2則、成功數1。
fake_requests應只有1筆，stream為True、store為False。虛構usage是測試數字，不代表真實token或費用。
通過後再跑後面的五項離線檢查；失敗時依例外回到A或B，不修改測試器。

In [ ]:
RUN_FAKE_STREAM_CHECK = False
if RUN_FAKE_STREAM_CHECK:
    fake_requests = []
    shown = []
    fake_state = new_chat_state()
    fake_client = make_fake_client(make_fake_events(), fake_requests)
    fake_text, fake_count = ask_chat_stream(
        fake_state, "離線問候", on_delta=shown.append, api_client=fake_client,
    )
    assert fake_text == "你好！" and fake_count == 2 and shown == ["你好", "！"]
    assert len(fake_state["history"]) == 2 and fake_state["completed_requests"] == 1
    assert len(fake_requests) == 1
    assert fake_requests[0]["stream"] is True and fake_requests[0]["store"] is False
    print("假串流：", fake_text, "｜delta數：", fake_count, "｜正式history：", fake_state["history"])

### 五項測試怎麼判讀？（對照run_offline_checks）
| 測試器原程式 | 檢查意義 | 若失敗先看 |
|---|---|---|
| requests[1]["input"] == state["history"][:3] | 第二輪帶第一輪配對與新user，並重送instructions | A1／ask_chat_once流程 |
| reset_chat後new_input只有本次user | 舊姓名不再送入模型；統計清零 | A2 |
| build_chat_input(..., 1) == history[-2:] + [...] | 最近一輪保持完整配對 | A1b–A1d |
| before與len(requests)比較 | 空白在本機擋下，沒有呼叫假create | 已提供require_text |
| 成功後兩則history；各失敗後state == snapshot | 完成才保存，失敗無半套正式紀錄 | B3–B5 |
snapshot以json往返建立獨立副本，避免與state共用內部list/dict；不是把對話永久存檔。
except RuntimeError表示測試預期失敗；else的AssertionError則表示「應失敗卻成功」。五行PASS不是模型品質分數。

JSON往返能為本教材的簡單state建立獨立副本；不是任意Python物件的深複製。`try/except/else`中的else只在沒有例外時執行，所以「應失敗卻成功」才拋AssertionError。這兩點供讀測試器，不另安排語法活動。


In [ ]:
def run_offline_checks():
    """完成練習後先驗證五個契約；不使用 API key，也不付費。"""
    rows = []
    requests = []
    state = new_chat_state()
    fake = make_fake_client(fake_response("姓名是小安"), requests)
    ask_chat_once(state, "我叫小安", api_client=fake)
    ask_chat_once(state, "我叫什麼？", api_client=fake)
    assert requests[1]["input"] == state["history"][:3]
    assert requests[1]["instructions"] == CHAT_INSTRUCTIONS
    rows.append("PASS：第二輪包含真實保存的前文")

    reset_chat(state)
    new_input = build_chat_input("我叫什麼？", state["history"])
    assert new_input == [{"role": "user", "content": "我叫什麼？"}]
    assert state["completed_requests"] == 0 and state["usage"]["total_tokens"] == 0
    rows.append("PASS：reset 後請求沒有舊姓名")

    history = [
        {"role": "user", "content": "舊問題"}, {"role": "assistant", "content": "舊答案"},
        {"role": "user", "content": "新問題"}, {"role": "assistant", "content": "新答案"},
    ]
    assert build_chat_input("追問", history, 1) == history[-2:] + [{"role": "user", "content": "追問"}]
    rows.append("PASS：最近N輪保持完整配對")

    before = len(requests)
    try:
        ask_chat_once(state, " ", api_client=fake)
    except ValueError:
        pass
    # 細節選讀：else只在try沒有例外時執行；此測試預期空白會拋錯。
    else:
        raise AssertionError("空白輸入沒有被擋下。")
    assert len(requests) == before and state["history"] == []
    rows.append("PASS：空白輸入不呼叫API")

    displayed = []
    stream_fake = make_fake_client(make_fake_events(), requests)
    text, count = ask_chat_stream(state, "你好", on_delta=displayed.append, api_client=stream_fake)
    assert text == "你好！" and count == 2 and "".join(displayed) == text
    assert len(state["history"]) == 2
    # 細節選讀：JSON往返為本教材的簡單state建立獨立副本；不是任意物件的通用深複製。
    snapshot = json.loads(json.dumps(state))
    for kind in ("error", "response.failed", "response.incomplete", "missing"):
        bad = make_fake_client(make_fake_events(kind), requests)
        try:
            ask_chat_stream(state, "失敗回合", on_delta=displayed.append, api_client=bad)
        except RuntimeError:
            pass
        else:
            raise AssertionError(f"未拒絕：{kind}")
        assert state == snapshot
    rows.append("PASS：串流成功才保存；失敗／缺完成事件維持原狀態")
    return rows


RUN_OFFLINE_CHECKS = False
if RUN_OFFLINE_CHECKS:
    for row in run_offline_checks():
        print(row)

In [ ]:
def run_live_chat_checks():
    """三次真實請求，印出請求與回答供人工覆核，不用猜測式答案assert。"""
    state = new_chat_state()
    for question in ["我叫小安，專題是課程FAQ。", "我的姓名與專題是什麼？"]:
        print("input：", build_chat_input(question, state["history"]))
        print(ask_chat_once(state, question))
    reset_chat(state)
    print("reset後input：", build_chat_input("我的姓名是什麼？", state["history"]))
    print(ask_chat_once(state, "我的姓名是什麼？"))
    print("reset後usage：", state["usage"])

# 真實測試：先完成A與離線檢查；3次API請求，確認後才開啟。
RUN_LIVE_TESTS = False
if RUN_LIVE_TESTS:
    run_live_chat_checks()

### 🧪 測試與修訂紀錄
| 條件 | 預期請求／狀態 | 實際 | 通過 | 修改理由 |
|---|---|---|---|---|
| 第二輪追問 | 帶第一輪user/assistant |  |  |  |
| reset | 不帶舊資料 |  |  |  |
| 最近N輪 | 完整配對＋本次user |  |  |  |
| 空白 | API請求數不變 |  |  |  |
| 串流中斷 | history、usage、成功數不變 |  |  |  |
| 真實回答 | 姓名／主題符合前文，reset後不聲稱記得 |  |  |  |
同儕覆核：對方能否從input指出模型看得到與看不到的內容？失敗是否留下半套紀錄？

## 9️⃣ 課後整合：簡易CLI骨架（選做）
課堂先用固定問題列表，避免互動input卡住「全部執行」。
完成A/B後才開啟CLI，支援/reset與/exit；失敗時保留原對話，提示重試。

In [ ]:
def run_console_chat():
    """重用手動history串流函式，不另寫第二套聊天邏輯。"""
    state = new_chat_state()
    while True:
        user_text = input("\n你：").strip()
        if user_text == "/exit":
            break
        if user_text == "/reset":
            reset_chat(state)
            print("本機對話已清空。")
            continue
        try:
            ask_chat_stream(state, user_text)
            print()
        # 細節選讀：tuple列出任一可接受的錯誤型別；不是同時發生兩種錯誤。
        except (ValueError, RuntimeError) as exc:
            print(f"\n本輪未成功：{exc}，可重試。")

# 完成練習並確認成本後，手動取消下一行註解；每輪都會付費。
# run_console_chat()

## 🔎 進階選讀：重要事實與類別封裝

重要偏好可另存本機state，作為清楚標示的使用者資料提供模型，不提升成可覆蓋instructions的指令；未寫檔或存資料庫，重啟後不保留，不稱永久記憶。
選做：另建preferred_language與project_topic，印出裁切後input確認仍含選定資料，說明reset是否清除它們；不要求額外付費實驗。

**下一格ManualChatSession只包裝已教函式，不另寫聊天邏輯：**

| 方法／屬性 | 對應函式版 |
|---|---|
| __init__中的self.state | new_chat_state，每個物件建立自己的state |
| ask | ask_chat_once，回傳str |
| ask_stream | ask_chat_stream，回傳文字與delta數 |
| reset | reset_chat(self.state)，回傳None |

self指目前物件；建立實例只初始化狀態，ask／ask_stream才呼叫API。類別不是本週必做，不需將A/B改寫為類別，也不增加繼承、super或nonlocal課程。


In [ ]:
class ManualChatSession:
    """選讀：state成為物件屬性，核心函式維持原樣。"""
    # 細節選讀：建立實例時初始化，每個實例各自呼叫new_chat_state；不呼叫API。
    def __init__(self, instructions=CHAT_INSTRUCTIONS, max_turns=5):
        self.instructions = instructions
        self.max_turns = max_turns
        self.state = new_chat_state()

    # 細節選讀：session.ask(...)會自動把目前實例傳給self，再轉交既有聊天函式。
    def ask(self, user_text):
        return ask_chat_once(self.state, user_text, self.max_turns, self.instructions)

    def ask_stream(self, user_text):
        return ask_chat_stream(self.state, user_text, self.max_turns, self.instructions)

    def reset(self):
        reset_chat(self.state)

# 只定義類別，不在此呼叫API。讀懂函式版後再比較self.state與普通state。

## ✅ 完成檢核與後續銜接
- [ ] 能說明人工範例、實際history與response ID的差別。
- [ ] 完成A：最近N輪與reset，印出真正送出的input。
- [ ] 完成B：delta累積、完成驗證、失敗不更新history。
- [ ] 固定instructions每輪送出，usage不等於完整帳單。
- [ ] 通過離線五項契約，記錄真實API尚未驗證的部分。
- [ ] 能區分模型答錯、輸入／輸出契約錯誤及API失敗。
第六周仍教Structured Outputs與資料抽取；第五周不提前塞入JSON Schema。
第七周把同一history放入st.session_state，最近五輪加本次user送入模型。

## 📝 課後任務與繳交規格
選課程FAQ、學習助教或校園資訊情境，沿用本週手動history函式，完成A/B。
| 繳交項目 | 具體內容 | 驗收方式 |
|---|---|---|
| Notebook | 完成A1/A2、B1–B5；保留測試器 | 函式可定義，離線檢查通過 |
| instructions | 身分、語言、回答限制、未知資料處理 | 每輪確實送出相同規則 |
| 對話紀錄 | 三輪含追問，另附reset後一輪 | 保存實際input與回答；不預填模型答案 |
| 最近N輪觀察 | 設N=1，印出輸入與本機history則數 | 完整配對＋本次user；裁切不刪原history |
| 串流與測試 | 片段、完整文字、delta數、五項PASS及失敗時state比較 | 完成才保存；未完成不污染state |
| 真實測試與usage | 標示已執行／未執行、成功回合用量 | 不把假usage當真實用量，不以成功數代替帳單 |
| 150字反思 | 本機狀態、保存時機、失敗重試與修正理由 | 以實際input／測試結果支持說明 |
若暫無金鑰，先交離線成果並標明真實API未驗證；真實回答紀錄仍是待補項目，不能拿假回應冒充。
CLI、類別與重要事實為延伸，不要求全部完成。安全：使用虛構資料，不提交API key、學生個資、成績或機密。

## 🤝 同儕覆核
交換input與測試表，確認每個角色的內容來源、裁切後仍為完整配對、失敗不留下半套state。
找一項未驗證或失敗條件，修改一個主要變因後重測並記錄原因。

## 📚 官方文件與版本使用
- [Conversation state](https://developers.openai.com/api/docs/guides/conversation-state)
- [Streaming responses](https://developers.openai.com/api/docs/guides/streaming-responses)
- [Responses API](https://developers.openai.com/api/reference/resources/responses/methods/create)
模型沿用OPENAI_MODEL／gpt-5.4-mini；本次沒有用教學帳號驗證可用性，課前須試跑。
本修訂版與原教材並存，課堂選定一組使用，不要求學生交錯執行兩版。